# Importations

In [1]:
# Numerical and scientific python programming
import numpy as np

from scipy.stats import special_ortho_group
from scipy.linalg import eigh

# Auxiliary python functions
from itertools import combinations, product
from functools import reduce
from typing import Sequence, Iterable

# Local importations
from moments.bloch import (compute_pauli_basis, compute_tensor_basis, compute_subset_index_map,
                           compute_bloch_vector, compute_dm_from_bloch, compute_bloch_norms_from_vector)
from moments.quantum import generate_rand_dm, compute_is_valid_dm


# Definitions

---

**Observation 1.** Any fully separable four-qubit state obeys

$$
2 S_4 + S_3 \le 2 + S_1 \, .
$$

---

---

**Observation 2.**

---

Whithin the purity constraints, Bloch lengths and Sector lengths satisfy:

- $||\vec r_k||^2 \in [0, 1]$, then $S_1 \in [0, 3]$,
- $||\vec r_{kl}||^2 \in [0, 3]$, then $S_2 \in [0, 9]$,
- $||\vec r_{klm}||^2 = S_3 \in [0, 7]$,
- $||\vec r_1234||^2 = S_4 \in [0, 15]$.

with $k, lm , = 1, 2, 3$. These bounds are not tight.

In [2]:
def compute_rand_SO_subset(subset_index_map):
    Q = {}
    for subset in subset_index_map.keys():
        Q[subset] = special_ortho_group.rvs(len(subset_index_map[subset]))
    return Q

def check_tensor_rot(Q):
    for subset in Q.keys():
        if len(subset) > 1:
            Q_tensor = np.array([1])
            for m in subset:
                Q_tensor = np.kron(Q_tensor, Q[(m,)])
            if np.allclose(Q[subset], Q_tensor):
                return False
    return True

def tensor_product(O_v: Sequence[np.ndarray]) -> np.ndarray:
    try:
        return reduce(np.kron, O_v)
    except ValueError as e:
        raise ValueError("Some arrays in O_v have incompatible shapes for Kronecker product.") from e

def compute_pt(dim: list[int], A: np.ndarray, subsystem: int | Iterable[int] = 1) -> np.ndarray:
    """
    Compute the partial transpose of a multipartite operator.

    Parameters
    ----------
    dim : list[int]
        Local Hilbert-space dimensions.
    A : ndarray
        Matrix of shape (prod(dim), prod(dim)).

    subsystem : int or iterable[int]
        Which subsystem(s) to transpose.

    Returns
    -------
    ndarray
        Partial transpose.
    """

    nsub = len(dim)
    D = np.prod(dim)

    if A.shape != (D, D):
        raise ValueError("Matrix shape inconsistent with dimensions.")

    if isinstance(subsystem, int):
        subsystem = [subsystem]
    subsystem = set(subsystem)

    tensor = A.reshape(*dim, *dim)

    perm = list(range(2 * nsub))

    for s in subsystem:
        perm[s], perm[s + nsub] = perm[s + nsub], perm[s]

    tensor_pt = tensor.transpose(perm)

    return tensor_pt.reshape(D, D)

def compute_tr_norm(eigenvalues: np.ndarray | None = None, A: np.ndarray | None = None) -> float:
    """
    Compute the trace norm of a Hermitian matrix.
    For a Hermitian matrix A, the trace norm reduces to the sum of absolute eigenvalues.

    Parameters
    ----------
    eigenvalues : np.ndarray, optional
        Precomputed eigenvalues of A. If provided, A is not used.
    A : np.ndarray, optional
        A Hermitian matrix of shape (d, d). Required if eigenvalues is not provided.
    
    Returns
    -------
    float
        The trace norm of A.
    """
    if eigenvalues is None:
        if A is None:
            raise ValueError("Provide eigenvalues or matrix.")
        if A.shape[0] != A.shape[1]:
            raise ValueError("Matrix must be square.")
        if not np.allclose(A, A.conj().T):
            raise ValueError("Matrix must be Hermitian.")
        eigvals = eigh(A, eigvals_only=True)
    
    return float(np.sum(np.abs(eigvals)))

def compute_negativity(rho: np.ndarray, dim: list[int], subsystem: int | Iterable[int] = 1) -> float:
    """
    Compute the entanglement negativity of a quantum state by specifying the bipartition.

    Parameters
    ----------
    rho : np.ndarray
        Density matrix of shape (dA*dB, dA*dB)
    dim : List[int]
        Local Hilbert space dimensions.
    subsystem : int
        Subsystem to partially transpose.

    Returns
    -------
    float
        The entanglement negativity.
    """
    rho_pt = compute_pt(dim, rho, subsystem)
    return (compute_tr_norm(A=rho_pt) - 1.) / 2.

In [3]:
dn, N = 2, 4
dim = [dn]*N
d = int(np.prod(dim))

pauli_basis = compute_pauli_basis()

local_bases = [pauli_basis.copy()] * N
local_basis_sizes = [len(basis) for basis in local_bases]

tensor_basis = compute_tensor_basis(local_bases)
subset_index_map = compute_subset_index_map(local_basis_sizes)

# Viability test

In [9]:
rho = generate_rand_dm(d, d)
r = compute_bloch_vector(tensor_basis, subset_index_map, rho)

print("Local Bloch-vector dimensions:")
print("One-party subsystems:", len(r[(1,)]))
print("Two-party subsystems:", len(r[(1,2)]))
print("Three-party subsystems:", len(r[(1,2,3)]))
print("Four-party subsystems:", len(r[(1,2,3,4)]))

Local Bloch-vector dimensions:
One-party subsystems: 3
Two-party subsystems: 9
Three-party subsystems: 27
Four-party subsystems: 81


In [4]:
is_valid = False
j = 0

while not is_valid:
    j += 1
    
    rho = generate_rand_dm(d, d)
    r = compute_bloch_vector(tensor_basis, subset_index_map, rho)

    Q = compute_rand_SO_subset(subset_index_map)
    if not check_tensor_rot(Q):
        print("Warning: Q_M is equal to the tensor product of Q_m for some M")
    
    r_rot = {subset: Q[subset] @ r[subset] for subset in subset_index_map.keys()}
    
    rho_rot = compute_dm_from_bloch(tensor_basis, subset_index_map, r_rot)
    is_valid, _ = compute_is_valid_dm(rho_rot)

print("Sucess:", is_valid)
print("Number of iterations:", j)


/Users/alfonso/Documents/projects/-2024--bloch-lengths/.venv/lib/python3.14/site-packages/numpy/linalg/_linalg.py:2406: RuntimeWarning: divide by zero encountered in det
  r = _umath_linalg.det(a, signature=signature)
/Users/alfonso/Documents/projects/-2024--bloch-lengths/.venv/lib/python3.14/site-packages/numpy/linalg/_linalg.py:2406: RuntimeWarning: overflow encountered in det
  r = _umath_linalg.det(a, signature=signature)
/Users/alfonso/Documents/projects/-2024--bloch-lengths/.venv/lib/python3.14/site-packages/numpy/linalg/_linalg.py:2406: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


KeyboardInterrupt: 

# Local identity rotations

$$
Q_n = \mathbb I_3 \, , \quad \forall n \in N
$$
$$
Q_{nm} = \mathbb I_9 \, , \quad \forall n, m \in N
$$

In [11]:
is_valid = False
j = 0

while not is_valid:
    j += 1

    rho = generate_rand_dm(d, d)
    r = compute_bloch_vector(tensor_basis, subset_index_map, rho)

    Q_1234 = special_ortho_group.rvs(81)
    if np.allclose(Q_1234, np.identity(81)):
        print("Warning: Q_1234 is equal to the identity.")
    
    r_rot = r.copy()
    r_rot[(1, 2, 3, 4)] = Q_1234 @ r_rot[(1, 2, 3, 4)]
    
    rho_rot = compute_dm_from_bloch(tensor_basis, subset_index_map, r_rot)
    
    is_valid, _ = compute_is_valid_dm(rho_rot)

print("Sucess:", is_valid)
print("Number of iterations:", j)

/Users/alfonso/Documents/projects/-2024--bloch-lengths/.venv/lib/python3.14/site-packages/numpy/linalg/_linalg.py:2406: RuntimeWarning: divide by zero encountered in det
  r = _umath_linalg.det(a, signature=signature)
/Users/alfonso/Documents/projects/-2024--bloch-lengths/.venv/lib/python3.14/site-packages/numpy/linalg/_linalg.py:2406: RuntimeWarning: overflow encountered in det
  r = _umath_linalg.det(a, signature=signature)
/Users/alfonso/Documents/projects/-2024--bloch-lengths/.venv/lib/python3.14/site-packages/numpy/linalg/_linalg.py:2406: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


KeyboardInterrupt: 

In [16]:
R = compute_bloch_norms_from_vector(r)
R_rot = compute_bloch_norms_from_vector(r_rot)

bloch_diff = {subset: np.allclose(r[subset], r_rot[subset]) for subset in subset_index_map}
length_diff = {subset: np.allclose(R[subset], R_rot[subset]) for subset in subset_index_map}
DN = abs(compute_tripartite_negativity(rho, dim) - compute_tripartite_negativity(rho_rot, dim))

print("One-body Bloch vectors are constant:", (bloch_diff[(1,)] and bloch_diff[(2,)]))
print("Two-body Bloch vector is constant:", bloch_diff[(1, 2)])
print("Three-body Bloch vector is constant:", bloch_diff[(1, 2, 3)])
print("Density matrix is constant:", np.allclose(rho, rho_rot))
print("Bloch lengths are constant:", all(list(length_diff.values())))
print(f"Tripartite negativity difference: {DN:.4f}")

One-body Bloch vectors are constant: True
Two-body Bloch vector is constant: True
Three-body Bloch vector is constant: False
Density matrix is constant: False
Bloch lengths are constant: True
Tripartite negativity difference: 0.0224


# Exact rotations

In [12]:
def iterate_k_nonzero_arrays(k, size, min_value = 1, max_value = 10):
    
    base_array = np.zeros(size)
    
    value_range = range(min_value, max_value + 1)
    
    for indices in combinations(range(size), k):
        
        for values in product(value_range, repeat = k):
            
            array = base_array.copy()
            array[list(indices)] = values
            
            yield array

In [24]:
z = np.array([0, 0, 1])
zz = tensor_product([z, z])
zzz = tensor_product([z, z, z])
zzzz = tensor_product([z, z, z, z])
a, b, c = 0.5, 0.5, 0.5
r = {(1,): a * z, (2,): a * z, (3,): a * z, (4,): a * z,
     (1, 2): b * zz, (1, 3): b * zz, (1, 4): b * zz, (2, 3): b * zz, (2, 4): b * zz, (3, 4): b * zz,
     (1, 2, 3): c * zzz, (1, 2, 4): c * zzz, (1, 3, 4): c * zzz, (2, 3, 4): c * zzz,
     (1, 2, 3, 4): zzzz}

rho = compute_dm_from_bloch(tensor_basis, subset_index_map, r)
is_valid, _ = compute_is_valid_dm(rho)

N = compute_negativity(rho, [4, 4])

print("Original state:\n")
print(rho)
print("\nIs a valid density matrix?", is_valid)
print("Tripartite negativity:", N)

Original state:

[[0.5625+0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
  0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
  0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j]
 [0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
  0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
  0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j]
 [0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
  0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
  0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j]
 [0.    +0.j 0.    +0.j 0.    +0.j 0.0625+0.j 0.    +0.j 0.    +0.j
  0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
  0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j]
 [0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
  0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
  0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j]
 [0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.0625+0.j


In [34]:
k = 4
vectors, indices = [], []

for j, r_1234 in enumerate(iterate_k_nonzero_arrays(k, size = 81, max_value = 2)):
    
    r_rot = r.copy()
    r_rot[(1, 2, 3, 4)] = r_1234 / np.linalg.norm(r_1234)
    rho_rot = compute_dm_from_bloch(tensor_basis, subset_index_map, r_rot)
    
    is_valid, _ = compute_is_valid_dm(rho_rot)
    if is_valid:
        print(f'\n# {j}:\nr_1234 = {r_1234}')
        vectors.append(r_1234)
        indices.append(j)


# 1200208:
r_1234 = [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 1. 0. 0. 0. 1.]

# 1200223:
r_1234 = [2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 2. 0. 0. 0. 2.]

# 1248864:
r_1234 = [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 1.]

# 1248879:
r_1234 = [2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 2. 0. 0. 0. 0. 0. 0

In [36]:
negativities = []

for j, r_1234 in zip(indices, vectors):
    
    N_rot = compute_negativity(rho_rot, [2, 8])
    print(f'\n# {j}')
    print(f"N = {N_rot:.4f}")
    negativities.append(N_rot)

j_max = np.argmax(np.array(negativities))

print("\nMaximum negativity")
print(f"N = {negativities[j_max]:.4f}")


# 1200208
N = 0.1579

# 1200223
N = 0.1579

# 1248864
N = 0.1579

# 1248879
N = 0.1579

# 1261072
N = 0.1579

# 1261087
N = 0.1579

# 2433568
N = 0.1579

# 2433583
N = 0.1579

# 2444784
N = 0.1579

# 2444799
N = 0.1579

# 2499872
N = 0.1579

# 2499887
N = 0.1579

# 2531552
N = 0.1579

# 2531567
N = 0.1579

# 3626464
N = 0.1579

# 3626479
N = 0.1579

# 3672544
N = 0.1579

# 3672559
N = 0.1579

# 3724224
N = 0.1579

# 3724239
N = 0.1579

# 3737312
N = 0.1579

# 3737327
N = 0.1579

# 4783744
N = 0.1579

# 4783759
N = 0.1579

# 4842864
N = 0.1579

# 4842879
N = 0.1579

# 4871104
N = 0.1579

# 4871119
N = 0.1579

# 4918704
N = 0.1579

# 4918719
N = 0.1579

# 5977536
N = 0.1579

# 5977551
N = 0.1579

# 6004048
N = 0.1579

# 6004063
N = 0.1579

# 6032080
N = 0.1579

# 6032095
N = 0.1579

# 6975152
N = 0.1579

# 6975167
N = 0.1579

# 7037312
N = 0.1579

# 7037327
N = 0.1579

# 7067184
N = 0.1579

# 7067199
N = 0.1579

# 7118032
N = 0.1579

# 7118047
N = 0.1579

# 8012304
N = 0.1579

# 8012319